In [1]:
# 1. 导入模块、读取配置和路径
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

_base = Path('/home/intern_fjq_2026').resolve()
_current = Path.cwd().resolve()
_candidates = [
    _base / 'Projects' / 'chinese-wwm-roberta',
    _base.parent / 'intern_fjq_2026' / 'Projects' / 'chinese-wwm-roberta',
    _current,
    _current.parent,
]
ROOT = next((path for path in _candidates if (path / 'src').is_dir()), None)
assert ROOT is not None, 'Repository root was not found'
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.activation_rank import (
    finalize_activation_rank_run,
    load_activation_rank_config,
    plot_rank_curves,
    plot_rank_heatmap,
    preflight_activation_rank,
    run_pilot_stage,
    run_rank_analysis_stage,
    run_rank_shards_stage,
    run_sample_stage,
    run_wo_mechanism_stage,
    validate_pilot_outputs,
    validate_rank_analysis_outputs,
    validate_wo_mechanism_outputs,
)

CONFIG_PATH = ROOT / 'configs' / 'activation_rank.yaml'
config = load_activation_rank_config(CONFIG_PATH)
RUN_DIR = Path(config['output']['run_directory']).expanduser().resolve()
display({'config': str(CONFIG_PATH), 'run_directory': str(RUN_DIR)})

{'config': '/home/intern_fjq_2026/Projects/chinese-wwm-roberta/configs/activation_rank.yaml',
 'run_directory': '/home/intern_fjq_2026/Projects/chinese-wwm-roberta/artifacts/checkpoint_activation_rank/runs/financial_reports_v1'}

In [2]:
# 2. 服务器、GPU、磁盘、checkpoint与文本preflight
preflight = preflight_activation_rank(config)
display(preflight)
assert preflight['status'].eq('ok').all(), preflight

,check,status,detail
0,text,ok,/home/intern_fjq_2026/Projects/chinese-wwm-rob...
1,checkpoint,ok,/home/intern_fjq_2026/Projects/chinese-wwm-rob...
2,tokenizer,ok,/home/intern_fjq_2026/Projects/chinese-wwm-rob...
3,disk_budget,ok,"{""estimated_peak"": 6442450944, ""free"": 5420560..."
4,cpu_runtime,ok,"{""configured_threads"": 8, ""effective_threads"":..."
5,gpu_runtime,ok,"{""requested_device"": ""cuda:1"", ""cuda"": true, ""..."


In [4]:
# 3. 构建或严格复用sample manifest
sample_dir = run_sample_stage(config)
sample_manifest = pd.read_parquet(sample_dir / 'sample_manifest.parquet')
display(sample_manifest.head())
display(sample_manifest.groupby('shard')['token_count'].agg(['count', 'sum']))

,sample_row,source_row,report_id,text_sha256,stable_order,shard,token_count,is_unique_text
0,0,346841,392db3cbaf986062aef667b53d6994da444f439fd98453...,468d24aa18e6dab2482f63abe663851ef7b60279373433...,8991769536629,5,4,True
1,1,434587,c177b5ba77aa9a09b3c23a9922ffde040348deafe9d8ea...,0d1f506e308adb49f8a189f231bbe2754cfa821e4ee3bc...,38137858979464,0,510,True
2,2,218975,aa1326815aa7290a446a0652d787aa1dffc90f590c8d4b...,7074ed77bd1921175bfdaf101fe72a978b1e2cb02eb6a4...,40860073821757,5,4,True
3,3,845838,aaf9db3452ece97636747d2fbf304041e7da5732e67d0b...,967792222fe09f4cf0c194313fa823b965d01f8afbe66e...,58351295589966,6,510,True
4,4,987663,f18291e784eb476780fa2ab47c35f60ee5c9c5f6a188b4...,e7f1fbe3bb6af2bc1545ac1a06ce2ca04488fd030f03f2...,66113139498634,2,510,True


,count,sum
shard,,
0,3499,1250032
1,3374,1250126
2,3424,1250242
3,3473,1250062
4,3401,1250460
5,3399,1250149
6,3381,1250012
7,3463,1250496


In [5]:
# 4. hooks、精度、batch size与吞吐pilot
pilot_dir = run_pilot_stage(config)
pilot_manifest = validate_pilot_outputs(pilot_dir)
display(pd.Series(pilot_manifest))
display(pd.read_parquet(pilot_dir / 'batch_benchmark.parquet'))
display(pd.read_parquet(pilot_dir / 'precision_metrics.parquet'))

/home/intern_fjq_2026/miniconda3/envs/nlp_fjq/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


schema_version                                        activation_rank_pilot_v1.0
checkpoint_sha256              b6de21972c150139d0407b8e8e92bed3db9f23f92a1c70...
tokenizer_sha256               53b05407adaa567cf2cb7b57a8d05b9ab399cbd40af55d...
text_source_sha256             f143f1070d1e83a5958f43c1b4b28f9f8ffed0688f47d8...
config_sha256                  c80e304edd4b6bf96829f5980cd55dd2adbcee659fa39b...
max_length                                                                   512
git_commit                              0c65a5f8540bbbb1b2a5a6b504abe792dec82740
activation_rank_code_sha256    e8b000596506e3a12ae15589c4872090a32039f2fbb958...
run_fingerprint                2f493baf28bf77c7f5a630501ae68a7fd4032f9881f21d...
created_at                                                   2026-08-25T09:45:18
selected_compute_dtype                                                   float16
pilot_valid_tokens                                                        502928
batch_size                  

,batch_size,status,peak_memory_bytes,total_memory_bytes,headroom_fraction,checksum,rows,valid_tokens,batches,seconds,valid_tokens_per_second,padding_ratio
0,64,ok,2741471232,85043904512,0.967764,5.042122e+05,128,44528,3,1.465420,30385.835518,0.032743
1,128,ok,5065426944,85043904512,0.940438,1.014196e+06,256,89484,3,2.019725,44305.041967,0.032904
2,256,ok,9710602240,85043904512,0.885817,2.041665e+06,512,179221,3,3.996309,44846.635605,0.033922
3,512,ok,19001609216,85043904512,0.776567,4.106788e+06,1024,362221,3,7.642295,47396.886387,0.033287


,dtype,rows,valid_tokens,batches,seconds,valid_tokens_per_second,padding_ratio,peak_memory_bytes,maximum_normalized_erank_difference,maximum_k99_difference,maximum_relative_spectrum_difference_above_1pct,passed
0,float32,1410,502928,3,10.207954,49268.249446,0.033218,19484496384,0.000000,0,0.000000,True
1,float16,1410,502928,3,5.113459,98353.779115,0.033218,9988630016,0.000165,1,0.002066,True
2,bfloat16,1410,502928,3,5.216611,96408.959019,0.033218,9988630016,0.001673,2,0.023174,False


In [6]:
# 5. 运行缺失的rank shards；已验证分片自动跳过
# kernel重启或某一shard失败后，只需重跑cell 1与本cell。
moments_dir = run_rank_shards_stage(config)
moments_manifest = json.loads((moments_dir / 'manifest.json').read_text(encoding='utf-8'))
display(pd.Series(moments_manifest))

RuntimeError: 主样本norm/过滤审计失败，必须重建pilot而非静默调整: {"maximum_relative_norm_mean_shift": 0.010553624430842769, "maximum_relative_norm_std_shift": 21531.890426989856, "maximum_filtered_fraction": 0.3178639079119918, "passed": false}

In [ ]:
# 6. 合并moments、FP64特征分解与稳定性/坍缩分析
analysis_dir = run_rank_analysis_stage(config)
display(pd.Series(validate_rank_analysis_outputs(analysis_dir)))
rank_metrics = pd.read_parquet(analysis_dir / 'rank_metrics.parquet')
compression_metrics = pd.read_parquet(analysis_dir / 'compression_metrics.parquet')
display(rank_metrics)
display(compression_metrics)

In [ ]:
# 7. W^O协方差恒等式、方向分解与反事实机制分析
mechanism_dir = run_wo_mechanism_stage(config)
display(pd.Series(validate_wo_mechanism_outputs(mechanism_dir)))
display(pd.read_parquet(mechanism_dir / 'wo_summary.parquet'))
display(pd.read_parquet(mechanism_dir / 'wo_mechanism.parquet').head(24))

In [ ]:
# 8. 图表、结论表与最终run manifest
completed_dir = finalize_activation_rank_run(config)
final_manifest = json.loads((completed_dir / 'manifest.json').read_text(encoding='utf-8'))
display(pd.Series(final_manifest))
display(plot_rank_heatmap(completed_dir / 'analysis'))
display(plot_rank_curves(completed_dir / 'analysis'))
display(pd.read_parquet(completed_dir / 'compression_metrics.parquet'))